## NB03 — Robustness to image degradation

**Provenance and a deliberate asymmetry in this notebook.** This notebook sources
its numbers two ways, and the reason is a real constraint on disk, stated here so
it is not mistaken for an inconsistency:

- The **CLEAN** column is recomputed live from the frozen per-sample score files
  `checkpoints/scores_test_{resnet50,densenet121,vit}.csv`, using the exact
  `per_attack_auc` function from NB01. By construction the CLEAN column here is
  identical to the NB01 baseline table.
- The three **degraded** columns are parsed from the canonical robustness sweep
  log `results/slurm_robust_360859.out`. They are not recomputed, because the
  sweep wrote every condition's per-sample scores to the same generic CSV path,
  so each model overwrote the previous one. The per-condition score CSVs on disk
  now hold only the last model run (ViT). The aggregate log is the only artefact
  that preserves all three models across all conditions, so it is the source of
  record for the degraded columns.

Both sources are read from disk at execution time. No degraded figure is typed by
hand into this notebook.

**What this notebook tests.** Whether the two baseline behaviours (the recoverable
face-swap signal and the inverted text-edit ranking) hold up as the input image
is progressively degraded by JPEG compression and Gaussian blur. This is an
inference-only probe. No model is retrained.

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Frozen CLEAN score files (for the live-recomputed CLEAN column).
SCORE_FILES = {
    "ResNet50":    "checkpoints/scores_test_resnet50.csv",
    "DenseNet121": "checkpoints/scores_test_densenet121.csv",
    "ViT-B/16":    "checkpoints/scores_test_vit.csv",
}

# Canonical robustness sweep log (for the degraded columns).
ROBUST_LOG = "results/slurm_robust_360859.out"

# Degradation ladder. Labels as they appear in the log's CHECKPOINT headers,
# mapped to the column names used in the tables.
CONDITIONS = [
    ("CLEAN",            "CLEAN"),
    ("JPEG80 BLUR0.5",   "JPEG80 / blur0.5"),
    ("JPEG60 BLUR1.0",   "JPEG60 / blur1.0"),
    ("JPEG40 BLUR1.5",   "JPEG40 / blur1.5"),
]

# Model label as it appears in the log's CHECKPOINT headers.
LOG_MODEL_KEY = {
    "ResNet50":    "resnet50_baseline",
    "DenseNet121": "densenet121_baseline",
    "ViT-B/16":    "vit_baseline",
}

### CLEAN column, recomputed live (identical to NB01)

`per_attack_auc` is copied verbatim from NB01. It scores one attack type against
the bonafide pool, using the model's forged-class probability. Recomputing CLEAN
here rather than reading it from the log guarantees this notebook's CLEAN column
matches the NB01 baseline exactly.

In [2]:
def per_attack_auc(df, attack):
    """ROC-AUC for one attack type versus the bonafide pool.
    Positives are rows of the given attack type; negatives are bonafide rows
    (attack_type == 'none'). p_forged is the model's forged-class score.
    """
    mask = df["attack_type"].isin([attack, "none"])
    sub = df[mask]
    y_true = (sub["attack_type"] == attack).astype(int)   # 1 = forgery, 0 = bonafide
    return roc_auc_score(y_true, sub["p_forged"])


clean_auc = {}
for model, path in SCORE_FILES.items():
    df = pd.read_csv(path)
    clean_auc[model] = {
        "face": per_attack_auc(df, "face"),
        "text": per_attack_auc(df, "text"),
    }
clean_auc

{'ResNet50': {'face': 0.9145111111111112, 'text': 0.1365151515151515},
 'DenseNet121': {'face': 0.8872222222222222, 'text': 0.15658288770053475},
 'ViT-B/16': {'face': 0.774, 'text': 0.19497504456327985}}

### Degraded columns, parsed from the sweep log

The log is a sequence of blocks headed `CHECKPOINT: <model>_baseline - <CONDITION>`.
The per-attack AUCs sit on the two lines after the
`=== Per attack type (AUC vs bonafide pool) ===` marker, in the form
`face  ...  ROC-AUC=0.xxxx` and `text  ...  ROC-AUC=0.xxxx`. The parser below walks
the file, tracks the current model and condition from each header, and captures
those two AUC values. Parsing the log directly, rather than transcribing it,
removes any transcription error and reads the numbers from disk at execution
time.

In [3]:
def parse_robustness_log(path):
    """Walk the sweep log. Return nested dict: model_key -> condition -> {face,text}."""
    header_re = re.compile(r"CHECKPOINT:\s+(\S+)\s+-\s+(.+?)\s*$")
    auc_re    = re.compile(r"^(face|text)\b.*ROC-AUC=([0-9.]+)")
    out = {}
    cur_model = cur_cond = None
    for line in open(path):
        h = header_re.search(line)
        if h:
            cur_model, cur_cond = h.group(1), h.group(2).strip()
            out.setdefault(cur_model, {}).setdefault(cur_cond, {})
            continue
        m = auc_re.match(line.strip())
        if m and cur_model is not None:
            out[cur_model][cur_cond][m.group(1)] = float(m.group(2))
    return out


parsed = parse_robustness_log(ROBUST_LOG)
# Show which model/condition blocks were found, as a sanity check.
{k: list(v.keys()) for k, v in parsed.items()}

{'resnet50_baseline': ['CLEAN',
  'JPEG80 BLUR0.5',
  'JPEG60 BLUR1.0',
  'JPEG40 BLUR1.5'],
 'densenet121_baseline': ['CLEAN',
  'JPEG80 BLUR0.5',
  'JPEG60 BLUR1.0',
  'JPEG40 BLUR1.5'],
 'vit_baseline': ['CLEAN',
  'JPEG80 BLUR0.5',
  'JPEG60 BLUR1.0',
  'JPEG40 BLUR1.5']}

### Assemble the robustness tables

Two tables, one per attack type. The CLEAN column is the live-recomputed value
(canonical, matching NB01). The three degraded columns are the parsed log values.
All values are shown at four decimal places, so the table is precision-uniform.

In [4]:
def build_table(attack):
    rows = []
    for model in SCORE_FILES:
        key = LOG_MODEL_KEY[model]
        row = {"Architecture": model}
        for log_label, col_name in CONDITIONS:
            if log_label == "CLEAN":
                row[col_name] = clean_auc[model][attack]          # live recompute
            else:
                row[col_name] = parsed[key][log_label][attack]    # parsed from log
        rows.append(row)
    return pd.DataFrame(rows).set_index("Architecture")

face_table = build_table("face")
text_table = build_table("text")

fmt = {c: "{:.4f}" for _, c in CONDITIONS}
print("Face-swap AUC by degradation condition")
display(face_table.style.format(fmt))
print("\nText-edit AUC by degradation condition")
display(text_table.style.format(fmt))

Face-swap AUC by degradation condition


,CLEAN,JPEG80 / blur0.5,JPEG60 / blur1.0,JPEG40 / blur1.5
Architecture,,,,
ResNet50,0.9145,0.9138,0.8978,0.8745
DenseNet121,0.8872,0.8864,0.8797,0.8637
ViT-B/16,0.7740,0.7736,0.7709,0.7709



Text-edit AUC by degradation condition


,CLEAN,JPEG80 / blur0.5,JPEG60 / blur1.0,JPEG40 / blur1.5
Architecture,,,,
ResNet50,0.1365,0.1345,0.1327,0.1440
DenseNet121,0.1566,0.1559,0.1595,0.1646
ViT-B/16,0.1950,0.1955,0.1977,0.1998


### Consistency check: recomputed CLEAN against the log's CLEAN

The displayed CLEAN column is the CSV recompute. The log also records its own
CLEAN AUC, computed in memory at full float precision during the sweep. The two
are compared below. They agree to three decimal places; where they differ in the
fourth place it is because the frozen scores CSV stores `p_forged` to six
decimals while the log's value was computed at full precision. The notebook
displays the CSV recompute as canonical, consistent with NB01.

In [5]:
check = []
for model in SCORE_FILES:
    key = LOG_MODEL_KEY[model]
    for attack in ("face", "text"):
        recomputed = clean_auc[model][attack]
        logged = parsed[key]["CLEAN"][attack]
        check.append({
            "Architecture": model,
            "Attack": attack,
            "CLEAN (recomputed, canonical)": round(recomputed, 4),
            "CLEAN (log)": round(logged, 4),
            "abs diff": round(abs(recomputed - logged), 4),
        })
pd.DataFrame(check).set_index(["Architecture", "Attack"])

CLEAN (recomputed, canonical)  CLEAN (log)  abs diff
Architecture Attack                                                      
ResNet50     face                           0.9145       0.9147    0.0002
             text                           0.1365       0.1362    0.0003
DenseNet121  face                           0.8872       0.8873    0.0001
             text                           0.1566       0.1566    0.0000
ViT-B/16     face                           0.7740       0.7740    0.0000
             text                           0.1950       0.1950    0.0000

### Findings

**Face degrades gracefully.** For all three architectures the face-swap AUC
declines slowly and monotonically as degradation increases, staying well above
chance at the harshest setting. The recoverable face signal is robust to
compression and blur.

**Text stays inverted.** For all three architectures the text-edit AUC remains
firmly below the 0.5 diagonal across every degradation condition. Degradation
does not push the inverted text ranking back toward chance in any way that would
change the finding; the sub-diagonal behaviour is stable. This confirms that the
text failure is not a fragile artefact of pristine input. It persists under the
kind of quality loss a document acquired in the field would carry.

The claim these results support is specific to the model class under test:
off-the-shelf, whole-image, ImageNet-pretrained classifiers invert on text
manipulation, and that inversion is stable under image degradation. It is not a
claim that text manipulation is universally undetectable.

**Degradation grid.** Blur is applied before JPEG compression, at native
resolution, before the 224x224 resize. Gaussian blur radius takes values 0.5,
1.0, 1.5; JPEG quality takes values 80, 60, 40.